# GPT From Scratch

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### Tiny Shakespeare - Input dataset

In [6]:
with open('input/shakespeare.txt', 'r', encoding= 'utf- 8') as f:
    text = f.read()

In [7]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [8]:
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hello'))
print(decode([46, 43, 50, 50, 53]))

[46, 43, 50, 50, 53]
hello


In [9]:
data = torch.tensor(encode(text), dtype= torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [10]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [11]:
batch_size = 4      # Number of blocks to be processed parallely
block_size = 8      # Maximum context window for our transformer

def split(dataset):
    data = train_data if dataset == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))  # Arg1-> range, not inclusive of the upper cap, Arg2 -> Size which is a tuple to allow n dimensional tensor output
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = split('train')
print(xb.shape)
print(xb)
print(yb.shape)
print(yb)

torch.Size([4, 8])
tensor([[17, 26, 30, 37,  1, 14, 27, 24],
        [53, 59, 56, 57,  8,  0,  0, 14],
        [ 6,  1, 47, 41, 63,  7, 41, 53],
        [32, 46, 43,  1, 57, 53, 52,  1]])
torch.Size([4, 8])
tensor([[26, 30, 37,  1, 14, 27, 24, 21],
        [59, 56, 57,  8,  0,  0, 14, 30],
        [ 1, 47, 41, 63,  7, 41, 53, 50],
        [46, 43,  1, 57, 53, 52,  1, 53]])


In [22]:
class BigramModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # Acts as a lookup table, same as the counts from bigram

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            # Batch, Time, and Channel
            B,T,C = logits.shape

            # cross_entropy expects the second dimension to be 'Channel' -> the number of fields used for embedding
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_tokens):
        for _ in range(max_tokens):
            logits, loss = self(idx)             # This invokes nn.Module call function which calls the forward function 
            logits = logits[:,-1,:]
            prob = F.softmax(logits, dim= -1)
            pred = torch.multinomial(prob, num_samples= 1)
            idx = torch.cat((idx, pred), dim= 1)
        return idx

In [24]:
m = BigramModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.6920, grad_fn=<NllLossBackward0>)

?rAB.tGtL.vB!. gdh..qABfiPPQB$XLyzbBzG:hwwoyZpsZWaV-Ba-;ee,hdqT;VX;W:mAGcaXZ'
Zn'sMPAH;?SDsKbS'-aYAs
